# Document Chunking Strategies in RAG
## Practical Guide: Understanding Pitfalls, Overlap, and Selecting Optimal Strategies

---

### Learning Roadmap:
1. **Part 1: The Flaw in Naive Character Chunking** (Mid-word & mid-sentence cuts)
2. **Part 2: What Overlap Solves** (Context continuity at boundaries & sliding window math)
3. **Part 3: Fixed-Size Token Chunking** (Why characters fail with embedding models & tokenizer limits)
4. **Part 4: Sentence & Paragraph-Based Recursive Splitting** (The industry baseline)
5. **Part 5: Chunking Structured Data** (Preserving Markdown tables & code blocks)
6. **Part 6: Semantic Chunking** (Detecting topic shifts via sentence embedding distances)
7. **Part 7: Real Retrieval Impact in ChromaDB** (Proving chunk quality with vector search)

---


## Part 1: The Core Problem with Naive Character Chunking

### Why Naive Chunking Fails in Production:
Naive character chunking divides a string blindly at every $N$ characters (`text[start : start + chunk_size]`).

Because it treats text as an arbitrary string of characters rather than words or semantic ideas:
1. **Mid-Word Splits:** Words like `Digital` get fractured into `"D"` (in Chunk 0) and `"igital"` (in Chunk 1).
2. **Mid-Sentence Splits:** Crucial conditional clauses (e.g., `"non-refundable after access is granted"`) get severed from their subjects.
3. **Embedding Distortion:** Embedding models (like `all-MiniLM-L6-v2`) encode word meaning. When fed partial characters or truncated grammar, their vector representations drift into irrelevant semantic space.

In [44]:
# 1. Define the naive fixed-character chunker (Zero overlap)
def fixed_character_chunker(text: str, chunk_size: int = 150) -> list[str]:
    """
    Splits text strictly every `chunk_size` characters.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size
    return chunks

# 2. Real-world Enterprise Policy Document (Refund & Expense Reimbursement)
policy_document = """Our refund policy allows customers to return products within 30 days of purchase.
The item must be in its original packaging and unused condition.
Digital downloads are excluded from this policy and are non-refundable after access.
To initiate a return, contact support@example.com with your order number.

For employee travel reimbursements, submit receipts within 14 business days.
Meals are reimbursed up to $75 per day with itemized food receipts.
Alcohol and personal entertainment expenses are strictly non-reimbursable.
"""

# 3. Run naive chunker with chunk_size = 150 characters
naive_chunks = fixed_character_chunker(policy_document, chunk_size=150)

print(f"Total document character length: {len(policy_document)}")
print(f"Total chunks produced: {len(naive_chunks)}\n")

for i, chunk in enumerate(naive_chunks):
    print("=" * 60)
    print(f"CHUNK {i} (Length: {len(chunk)} chars):")
    print(repr(chunk))  # repr shows exact characters, newlines, and trailing cuts
    print("-" * 40)
    print(chunk)


Total document character length: 527
Total chunks produced: 4

CHUNK 0 (Length: 150 chars):
'Our refund policy allows customers to return products within 30 days of purchase.\nThe item must be in its original packaging and unused condition.\nDig'
----------------------------------------
Our refund policy allows customers to return products within 30 days of purchase.
The item must be in its original packaging and unused condition.
Dig
CHUNK 1 (Length: 150 chars):
'ital downloads are excluded from this policy and are non-refundable after access.\nTo initiate a return, contact support@example.com with your order nu'
----------------------------------------
ital downloads are excluded from this policy and are non-refundable after access.
To initiate a return, contact support@example.com with your order nu
CHUNK 2 (Length: 150 chars):
'mber.\n\nFor employee travel reimbursements, submit receipts within 14 business days.\nMeals are reimbursed up to $75 per day with itemized food receipts'
-

## Part 2: What Overlap Solves (The Boundary Problem & Sliding Window Math)

### 1. The Core Purpose of Overlap:
When chunks are sliced with **zero overlap**, any concept, clause, or entity that happens to fall across the split line is sheared in half.
* A question asked by a user rarely aligns with our arbitrary slicing points.
* **Chunk Overlap (Sliding Window)** forces each subsequent chunk to rewind by $M$ units before taking the next cut.

### 2. The Sliding Window Math:
- **`chunk_size` ($L$):** Maximum length of each chunk.
- **`chunk_overlap` ($O$):** How many characters/tokens to share with the preceding chunk.
- **`stride` ($S$):** The step size the window moves forward:  
  $$\text{stride} = \text{chunk\_size} - \text{chunk\_overlap}$$
- **Strict Invariant:** $\text{chunk\_overlap} < \text{chunk\_size}$. If $\text{chunk\_overlap} \ge \text{chunk\_size}$, $\text{stride} \le 0$, resulting in an infinite loop.

### 3. Industry Rule of Thumb:
Aim for **10% to 20% overlap** (e.g., 50 tokens on 400-token chunks, or 30-40 characters on 150-character chunks).
- **Too little overlap (< 5%):** Boundary severance still occurs.
- **Too much overlap (> 35%):** Explodes vector database storage and causes redundant duplicate chunks in retrieval.

In [45]:
# ==============================================================================
# PART 2: SLIDING WINDOW CHUNKER (WITH OVERLAP)
# ==============================================================================

def sliding_window_character_chunker(text: str, chunk_size: int = 150, chunk_overlap: int = 35) -> list[str]:
    """
    Splits text using a sliding window with overlap.
    Stride = chunk_size - chunk_overlap
    """
    assert chunk_overlap < chunk_size, "chunk_overlap must be strictly less than chunk_size!"
    
    chunks = []
    start = 0
    stride = chunk_size - chunk_overlap
    
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += stride
        
    return chunks

# Run chunker with 150 char size and 35 char overlap (approx 23% overlap)
overlap_chunks = sliding_window_character_chunker(policy_document, chunk_size=150, chunk_overlap=40)

print(f"Document Length: {len(policy_document)} chars")
print(f"Naive (0 overlap) chunks count : {len(naive_chunks)}")
print(f"Sliding Window (35 overlap) count: {len(overlap_chunks)}\n")

for i, chunk in enumerate(overlap_chunks):
    print("=" * 70)
    print(f"CHUNK {i} (Length: {len(chunk)} chars):")
    print(f"Raw repr : {repr(chunk)}")
    print("-" * 70)
    print(chunk)


Document Length: 527 chars
Naive (0 overlap) chunks count : 4
Sliding Window (35 overlap) count: 5

CHUNK 0 (Length: 150 chars):
Raw repr : 'Our refund policy allows customers to return products within 30 days of purchase.\nThe item must be in its original packaging and unused condition.\nDig'
----------------------------------------------------------------------
Our refund policy allows customers to return products within 30 days of purchase.
The item must be in its original packaging and unused condition.
Dig
CHUNK 1 (Length: 150 chars):
Raw repr : 'inal packaging and unused condition.\nDigital downloads are excluded from this policy and are non-refundable after access.\nTo initiate a return, contac'
----------------------------------------------------------------------
inal packaging and unused condition.
Digital downloads are excluded from this policy and are non-refundable after access.
To initiate a return, contac
CHUNK 2 (Length: 150 chars):
Raw repr : 'ter access.\nTo initiate 

## Part 3: Fixed-Size Token Chunking

### 1. Why Do We Need Token-Based Chunking?
LLMs and Embedding models do **not** process raw characters. They process **Tokens** (sub-word representations):
- 1 token ≈ 4 characters in English (~0.75 words per token).
- **Embedding Model Token Limits:**
  - `sentence-transformers/all-MiniLM-L6-v2`: Hard limit of **256 tokens**.
  - `text-embedding-3-small` (OpenAI): Limit of **8,191 tokens**.
- **The Character Risk:** If you define `chunk_size = 1000 characters`, in code snippets, legal text, or non-English languages, 1000 characters might explode into 400+ tokens, getting silently truncated by the embedding model!

### 2. How True Token Chunking Works:
1. **Encode:** Convert string text into integer token IDs using a tokenizer (e.g. `cl100k_base` via `tiktoken`).
2. **Slice:** Slice the **list of token IDs** using `stride = chunk_size_tokens - overlap_tokens`.
3. **Decode:** Convert each slice of token IDs back into readable text string via `encoding.decode()`.

> ⚠️ **Case Study (Class Notebook Bug Alert):**  
> In some naive implementations, developers write `while start < len(text): chunk = text[start:end]`. Even though they imported `tiktoken`, they sliced characters by accident! A **True Token Chunker** must slice the integer token ID list directly.

In [46]:
# ==============================================================================
# STEP 3A: INSPECTING TOKENS WITH TIKTOKEN (BPE ENCODING)
# ==============================================================================

import tiktoken

# Load OpenAI cl100k_base tokenizer (used by text-embedding-3-small / GPT-4)
encoding = tiktoken.get_encoding("cl100k_base")

doc_tokens = encoding.encode(policy_document)

print(f"Document Character Count: {len(policy_document)}")
print(f"Document Token Count    : {len(doc_tokens)}")
print(f"Average chars per token : {len(policy_document) / len(doc_tokens):.2f}\n")

# Let us inspect the first 15 tokens to see how words are tokenized
print("Token ID -> Decoded String Token:")
print("-" * 40)
for token_id in doc_tokens[:15]:
    token_str = encoding.decode([token_id])
    print(f"{token_id:<6} -> {repr(token_str)}")


Document Character Count: 527
Document Token Count    : 102
Average chars per token : 5.17

Token ID -> Decoded String Token:
----------------------------------------
8140   -> 'Our'
21639  -> ' refund'
4947   -> ' policy'
6276   -> ' allows'
6444   -> ' customers'
311    -> ' to'
471    -> ' return'
3956   -> ' products'
2949   -> ' within'
220    -> ' '
966    -> '30'
2919   -> ' days'
315    -> ' of'
7782   -> ' purchase'
627    -> '.\n'


In [47]:
# ==============================================================================
# STEP 3B: THE TRUE TOKEN CHUNKER WITH SLIDING WINDOW
# ==============================================================================

def true_token_chunker(
    text: str, 
    chunk_size_tokens: int = 40, 
    overlap_tokens: int = 10, 
    encoding_name: str = "cl100k_base"
) -> list[dict]:
    """
    Slices text by integer token IDs, guaranteeing strict token limit compliance.
    Returns a list of dicts with token_ids, token_count, and decoded_text.
    """
    assert overlap_tokens < chunk_size_tokens, "overlap_tokens must be strictly less than chunk_size_tokens!"
    
    enc = tiktoken.get_encoding(encoding_name)
    token_ids = enc.encode(text)
    stride = chunk_size_tokens - overlap_tokens
    
    chunks = []
    start = 0
    while start < len(token_ids):
        end = start + chunk_size_tokens
        chunk_token_ids = token_ids[start:end]
        decoded_text = enc.decode(chunk_token_ids)
        
        chunks.append({
            "token_count": len(chunk_token_ids),
            "char_count": len(decoded_text),
            "text": decoded_text
        })
        start += stride
        
    return chunks

# Run true token chunker: chunk_size=40 tokens, overlap=10 tokens (25% overlap)
token_chunks = true_token_chunker(policy_document, chunk_size_tokens=40, overlap_tokens=10)

print(f"Total Token Chunks Produced: {len(token_chunks)}\n")

for i, chunk in enumerate(token_chunks):
    print("=" * 70)
    print(f"CHUNK {i} | Tokens: {chunk['token_count']} | Characters: {chunk['char_count']}")
    print(f"Raw repr : {repr(chunk['text'])}")
    print("-" * 70)
    print(chunk['text'])


Total Token Chunks Produced: 4

CHUNK 0 | Tokens: 40 | Characters: 217
Raw repr : 'Our refund policy allows customers to return products within 30 days of purchase.\nThe item must be in its original packaging and unused condition.\nDigital downloads are excluded from this policy and are non-refundable'
----------------------------------------------------------------------
Our refund policy allows customers to return products within 30 days of purchase.
The item must be in its original packaging and unused condition.
Digital downloads are excluded from this policy and are non-refundable
CHUNK 1 | Tokens: 40 | Characters: 214
Raw repr : ' excluded from this policy and are non-refundable after access.\nTo initiate a return, contact support@example.com with your order number.\n\nFor employee travel reimbursements, submit receipts within 14 business days'
----------------------------------------------------------------------
 excluded from this policy and are non-refundable after access.
To

### Step 3C: Cross-Model Tokenizer Comparison (OpenAI vs MiniLM)

**Key Engineering Insight:**
Different embedding models use different tokenizer vocabularies and algorithms:
- **OpenAI (`cl100k_base`):** Byte-Pair Encoding (BPE) with **100,277 vocabulary tokens**.
- **MiniLM (`all-MiniLM-L6-v2`):** WordPiece with **30,522 vocabulary tokens**.

Because MiniLM has a smaller vocabulary, modern or compound words (like `frontend` and `microservices`) get broken down into smaller sub-words with the `##` prefix (indicating a continuation morpheme).

> 🚨 **Golden Rule for RAG:** Always chunk documents using the **exact tokenizer** of the embedding model that will generate vectors, otherwise silent token truncation will occur!

In [48]:
# ==============================================================================
# STEP 3C: COMPARING OPENAI (BPE) VS MINILM (WORDPIECE) TOKENIZATION
# ==============================================================================

import tiktoken
from transformers import AutoTokenizer

# 1. Initialize both tokenizers
openai_enc = tiktoken.get_encoding("cl100k_base")
minilm_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

# 2. Test words that expose vocabulary differences
test_words = ["Digital", "frontend", "microservices", "reimbursements", "non-refundable"]

print("=" * 80)
print(f"{'WORD':<18} | {'OPENAI TOKENS (cl100k_base)':<28} | {'MINILM TOKENS (WordPiece)':<30}")
print("=" * 80)

for word in test_words:
    # OpenAI BPE tokens
    openai_toks = [openai_enc.decode([t]) for t in openai_enc.encode(word)]
    
    # MiniLM WordPiece tokens (## denotes sub-word continuation)
    minilm_toks = minilm_tokenizer.tokenize(word)
    
    print(f"{word:<18} | {str(openai_toks):<28} | {str(minilm_toks):<30}")

print("=" * 80)

# 3. Compare total token counts on our entire policy document
doc_openai_tokens = len(openai_enc.encode(policy_document))
doc_minilm_tokens = len(minilm_tokenizer.encode(policy_document, add_special_tokens=False))

print(f"\nEntire Policy Document Token Count:")
print(f"  • OpenAI Tokens : {doc_openai_tokens} tokens")
print(f"  • MiniLM Tokens : {doc_minilm_tokens} tokens  <-- (+{doc_minilm_tokens - doc_openai_tokens} more tokens due to smaller 30k vocab!)")


WORD               | OPENAI TOKENS (cl100k_base)  | MINILM TOKENS (WordPiece)     
Digital            | ['Digital']                  | ['digital']                   
frontend           | ['frontend']                 | ['front', '##end']            
microservices      | ['micro', 'services']        | ['micro', '##ser', '##vic', '##es']
reimbursements     | ['re', 'im', 'burse', 'ments'] | ['rei', '##mb', '##urse', '##ments']
non-refundable     | ['non', '-ref', 'und', 'able'] | ['non', '-', 'ref', '##unda', '##ble']

Entire Policy Document Token Count:
  • OpenAI Tokens : 102 tokens
  • MiniLM Tokens : 107 tokens  <-- (+5 more tokens due to smaller 30k vocab!)


## Part 4: Sentence-Based, Paragraph-Based & Recursive Splitting

### 1. The Core Limitation of Fixed Chunking (Why We Need Structure):
In Part 3, Token Chunking stopped words from splitting, but **sentences were still severed mid-thought** because the chunker counted tokens blindly without caring about grammar.

Human writing is naturally hierarchical:
- **Paragraphs (`\n\n`):** Represent complete topical ideas.
- **Sentences (`. `):** Represent complete factual thoughts.
- **Words (` `):** The fundamental vocabulary units.

### 2. The Production Standard: `RecursiveCharacterTextSplitter`
Instead of cutting text blindly at character $N$ or token $M$, the **Recursive Splitter** tests separators in priority order:
```text
Priority 1: "\n\n" (Keep entire paragraphs intact if they fit inside chunk_size)
Priority 2: "\n"   (If paragraph exceeds chunk_size, split by lines)
Priority 3: ". "   (If line exceeds chunk_size, split at sentence full stops)
Priority 4: " "    (Last resort: split at word boundaries)
```
🏆 **Verdict:** This is the undisputed baseline strategy powering **80% of enterprise RAG pipelines** today.

In [49]:
# ==============================================================================
# STEP 4A: PURE PARAGRAPH-BASED & SENTENCE-BASED SPLITTING
# ==============================================================================

# 1. Paragraph-based split (Splits strictly by blank lines / \n\n)
paragraph_chunks = [p.strip() for p in policy_document.split("\n\n") if p.strip()]

print(f"Total Paragraphs Found: {len(paragraph_chunks)}")
for i, para in enumerate(paragraph_chunks):
    print(f"\n--- PARAGRAPH {i} (Length: {len(para)} chars) ---")
    print(para)

# 2. Sentence-based split (Splits strictly at full stops)
# Each sentence is a standalone factual assertion
sentence_chunks = [s.strip() for s in policy_document.replace("\n", " ").split(". ") if s.strip()]

print(f"\nTotal Sentences Found: {len(sentence_chunks)}")
for i, sent in enumerate(sentence_chunks[:4]):
    print(f"  Sentence {i}: {sent}...")


Total Paragraphs Found: 2

--- PARAGRAPH 0 (Length: 305 chars) ---
Our refund policy allows customers to return products within 30 days of purchase.
The item must be in its original packaging and unused condition.
Digital downloads are excluded from this policy and are non-refundable after access.
To initiate a return, contact support@example.com with your order number.

--- PARAGRAPH 1 (Length: 219 chars) ---
For employee travel reimbursements, submit receipts within 14 business days.
Meals are reimbursed up to $75 per day with itemized food receipts.
Alcohol and personal entertainment expenses are strictly non-reimbursable.

Total Sentences Found: 7
  Sentence 0: Our refund policy allows customers to return products within 30 days of purchase...
  Sentence 1: The item must be in its original packaging and unused condition...
  Sentence 2: Digital downloads are excluded from this policy and are non-refundable after access...
  Sentence 3: To initiate a return, contact support@example.

In [50]:
# ==============================================================================
# STEP 4B: LANGCHAIN RECURSIVE CHARACTER TEXT SPLITTER (THE INDUSTRY STANDARD)
# ==============================================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the recursive splitter
# It attempts splits at ["\n\n", "\n", ". ", " "] in strict order
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,      # Target chunk size
    chunk_overlap=30,    # Stride overlap to maintain boundary context
    separators=["\n\n", "\n", ". ", " "]
)

recursive_chunks = recursive_splitter.split_text(policy_document)

print(f"Total Recursive Chunks: {len(recursive_chunks)}\n")

for i, chunk in enumerate(recursive_chunks):
    print("=" * 70)
    print(f"CHUNK {i} (Length: {len(chunk)} chars):")
    print(f"Raw repr : {repr(chunk)}")
    print("-" * 70)
    print(chunk)


Total Recursive Chunks: 4

CHUNK 0 (Length: 146 chars):
Raw repr : 'Our refund policy allows customers to return products within 30 days of purchase.\nThe item must be in its original packaging and unused condition.'
----------------------------------------------------------------------
Our refund policy allows customers to return products within 30 days of purchase.
The item must be in its original packaging and unused condition.
CHUNK 1 (Length: 158 chars):
Raw repr : 'Digital downloads are excluded from this policy and are non-refundable after access.\nTo initiate a return, contact support@example.com with your order number.'
----------------------------------------------------------------------
Digital downloads are excluded from this policy and are non-refundable after access.
To initiate a return, contact support@example.com with your order number.
CHUNK 2 (Length: 144 chars):
Raw repr : 'For employee travel reimbursements, submit receipts within 14 business days.\nMeals are reim

## Part 5: Chunking Structured Data (Markdown Tables & Code)

### 1. The Structured Content Crisis:
Standard text splitters are designed for paragraphs of prose. When applied to **Tables** or **Code**:
1. **The Orphan Row Disaster:** A table split in half separates the data rows from the column header row. A chunk containing `| Wellness | $500 | Yes |` becomes completely unintelligible because the LLM doesn't know if `$500` is daily, monthly, or annual!
2. **Broken Code Syntax:** Code cut mid-function retrieves incomplete snippets with syntax errors (`IndentationError`) that LLMs cannot evaluate.

### 2. The Solution (Structure-Aware Chunking):
- **For Tables:** Keep the entire table intact as an atomic unit using paragraph boundaries (`\n\n`), or prepend the table header to every split chunk.
- **For Code:** Use syntax-aware splitters that break at `class` and `def` function boundaries.

In [51]:
# ==============================================================================
# STEP 5A: NAIVE SPLIT VS STRUCTURE-AWARE TABLE SPLITTING
# ==============================================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter

employee_allowance_doc = """Here is the official employee benefit policy for remote engineers:

| Allowance Type | Annual Cap | Approval Required | Receipts Needed |
|---|---|---|---|
| Home Office Setup | $1,000 | Manager | Yes |
| Wellness & Gym | $500 | Automatic | Yes |
| Learning & Books | $1,500 | Director | Yes |
| Mobile Phone Plan | $600 | Automatic | No |

Please submit all expense claims before the 25th of each calendar month.
"""

# 1. NAIVE SPLIT: Blindly splits by character count (e.g. 130 chars)
naive_table_splitter = RecursiveCharacterTextSplitter(chunk_size=130, chunk_overlap=0)
naive_table_chunks = naive_table_splitter.split_text(employee_allowance_doc)

print("❌ NAIVE SPLIT (The Broken Table Problem):")
for i, chunk in enumerate(naive_table_chunks):
    print(f"\n[CHUNK {i}]:")
    print(chunk)

print("\n" + "=" * 75 + "\n")

# 2. STRUCTURE-AWARE SPLIT: Uses paragraph boundaries and safe size
# Keeps the table intact as a single complete semantic block
structure_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400, 
    chunk_overlap=0,
    separators=["\n\n", "\n"]
)
structured_table_chunks = structure_splitter.split_text(employee_allowance_doc)

print("✅ STRUCTURE-AWARE SPLIT (Atomic Table Preserved):")
for i, chunk in enumerate(structured_table_chunks):
    print(f"\n[CHUNK {i}]:")
    print(chunk)


❌ NAIVE SPLIT (The Broken Table Problem):

[CHUNK 0]:
Here is the official employee benefit policy for remote engineers:

[CHUNK 1]:
| Allowance Type | Annual Cap | Approval Required | Receipts Needed |
|---|---|---|---|

[CHUNK 2]:
| Home Office Setup | $1,000 | Manager | Yes |
| Wellness & Gym | $500 | Automatic | Yes |

[CHUNK 3]:
| Learning & Books | $1,500 | Director | Yes |
| Mobile Phone Plan | $600 | Automatic | No |

[CHUNK 4]:
Please submit all expense claims before the 25th of each calendar month.


✅ STRUCTURE-AWARE SPLIT (Atomic Table Preserved):

[CHUNK 0]:
Here is the official employee benefit policy for remote engineers:

| Allowance Type | Annual Cap | Approval Required | Receipts Needed |
|---|---|---|---|
| Home Office Setup | $1,000 | Manager | Yes |
| Wellness & Gym | $500 | Automatic | Yes |
| Learning & Books | $1,500 | Director | Yes |
| Mobile Phone Plan | $600 | Automatic | No |

[CHUNK 1]:
Please submit all expense claims before the 25th of each calendar mont

In [52]:
# ==============================================================================
# STEP 5B: CODE-AWARE SPLITTING (SPLITTING BY FUNCTION & CLASS BOUNDARIES)
# ==============================================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter, Language

sample_python_code = """
import os

class PaymentGateway:
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.is_connected = False

    def connect(self):
        # Establish connection to Stripe API
        if self.api_key:
            self.is_connected = True
        return self.is_connected

    def process_refund(self, order_id: str, amount: float) -> bool:
        # Process customer refund
        print(f"Refunding {amount} for order {order_id}")
        return True
"""

# Python-specific splitter: knows about "def ", "class ", etc.
code_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=200,
    chunk_overlap=0
)

code_chunks = code_splitter.split_text(sample_python_code)
print(f"Total Code Chunks Produced: {len(code_chunks)}")
for i, chunk in enumerate(code_chunks):
    print(f"\n--- CODE CHUNK {i} ---")
    print(chunk.strip())


Total Code Chunks Produced: 4

--- CODE CHUNK 0 ---
import os

--- CODE CHUNK 1 ---
class PaymentGateway:
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.is_connected = False

--- CODE CHUNK 2 ---
def connect(self):
        # Establish connection to Stripe API
        if self.api_key:
            self.is_connected = True
        return self.is_connected

--- CODE CHUNK 3 ---
def process_refund(self, order_id: str, amount: float) -> bool:
        # Process customer refund
        print(f"Refunding {amount} for order {order_id}")
        return True


## Part 6: Semantic Chunking (Meaning & Topic-Shift Based)

### 1. What is Semantic Chunking?
When a document has **NO formatting, NO headers, and NO paragraph breaks** (e.g. continuous transcripts, meeting recordings, raw OCR scans), traditional splitters fail because they cannot detect invisible topic shifts.

**Semantic Chunking** uses an **Embedding Model** to inspect the meaning of each sentence:
1. Slices text into individual sentences: $S_0, S_1, S_2, \dots$
2. Embeds each sentence into a vector: $v_i = \text{Embed}(S_i)$
3. Computes the **Cosine Distance** between consecutive sentences: $\text{Distance} = 1 - \text{CosineSimilarity}(v_i, v_{i+1})$
4. When distance spikes above a threshold (a **Breakpoint**), it splits the document into a new chunk!

> 💡 While higher-level frameworks like LangChain provide wrapper splitters, implementing the vector math and cosine distance calculations from scratch provides deep insight into how topic shifts are mathematically detected.

In [53]:
# ==============================================================================
# STEP 6A: SEMANTIC CHUNKING FROM SCRATCH (USING ALL-MINILM-L6-V2)
# ==============================================================================

import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load local embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# A continuous unformatted document with 3 DIFFERENT TOPICS and ZERO paragraph breaks!
unformatted_document = (
    "Customers can return physical products within 30 days of delivery. "
    "Items must be undamaged and in original retail packaging. "
    "Digital software licenses and gift cards are strictly non-refundable. "
    "Employees traveling for business may expense meals up to 75 dollars per day. "
    "Hotel room bookings must be made through the approved corporate travel portal. "
    "All engineers must wear their security ID badges at all times in the building. "
    "Visitors must sign the guest register at the front desk before entering."
)

# 1. Split into individual sentences
sentences = [s.strip() for s in unformatted_document.split(". ") if s.strip()]
print(f"Total sentences to analyze: {len(sentences)}\n")

# 2. Embed all sentences
embeddings = model.encode(sentences)

# 3. Calculate Cosine Distance between consecutive sentences (Distance = 1 - Similarity)
distances = []
print("Consecutive Sentence Semantic Distances:")
print("=" * 75)
for i in range(len(sentences) - 1):
    sim = cosine_similarity([embeddings[i]], [embeddings[i+1]])[0][0]
    dist = 1.0 - sim
    distances.append(dist)
    print(f"S{i} -> S{i+1} | Distance: {dist:.4f} | Sim: {sim:.4f}")
    print(f"   S{i}   : {sentences[i][:50]}...")
    print(f"   S{i+1} : {sentences[i+1][:50]}...\n")

# 4. Breakpoint Detection: Split when distance spikes above 0.80
distance_threshold = 0.80
breakpoint_indices = [i + 1 for i, d in enumerate(distances) if d > distance_threshold]
print("=" * 75)
print(f"Detected Topic Shift Breakpoints at sentence indices: {breakpoint_indices}\n")

# 5. Build final semantic chunks
semantic_chunks = []
start = 0
for b_idx in breakpoint_indices:
    semantic_chunks.append(". ".join(sentences[start:b_idx]) + ".")
    start = b_idx
semantic_chunks.append(". ".join(sentences[start:]))

print(f"Total Semantic Chunks Produced: {len(semantic_chunks)}")
for i, chunk in enumerate(semantic_chunks):
    print("=" * 75)
    print(f"SEMANTIC CHUNK {i}:")
    print(chunk)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17076.30it/s]


Total sentences to analyze: 7

Consecutive Sentence Semantic Distances:
S0 -> S1 | Distance: 0.6314 | Sim: 0.3686
   S0   : Customers can return physical products within 30 d...
   S1 : Items must be undamaged and in original retail pac...

S1 -> S2 | Distance: 0.6618 | Sim: 0.3382
   S1   : Items must be undamaged and in original retail pac...
   S2 : Digital software licenses and gift cards are stric...

S2 -> S3 | Distance: 0.9186 | Sim: 0.0814
   S2   : Digital software licenses and gift cards are stric...
   S3 : Employees traveling for business may expense meals...

S3 -> S4 | Distance: 0.6623 | Sim: 0.3377
   S3   : Employees traveling for business may expense meals...
   S4 : Hotel room bookings must be made through the appro...

S4 -> S5 | Distance: 0.8402 | Sim: 0.1598
   S4   : Hotel room bookings must be made through the appro...
   S5 : All engineers must wear their security ID badges a...

S5 -> S6 | Distance: 0.7697 | Sim: 0.2303
   S5   : All engineers must wear their s

## Part 7: Real Retrieval Impact in ChromaDB (Proving Chunk Quality with Vector Search)

### 1. Does Chunking Actually Affect Search Quality?
Chunking is not just a theoretical pre-processing step — it directly determines **Retrieval Precision** and **LLM Hallucination Risk**:
- **Large / Bad Chunks (Setup A):** Drags along dozens of lines of irrelevant information (e.g., shipping costs and metro courier timelines when the user asked about return policies). This pollutes the LLM context window with noise.
- **Optimal Sized Chunks (Setup B):** Focuses precisely on the atomic rule requested, yielding a closer vector distance match and zero contextual noise.

### 2. Live Vector Search Experiment:
We index a mixed policy document into two separate **ChromaDB** collections and compare the exact passages returned for the user query:  
*`"How do I return a digital download?"`*

In [54]:
# ==============================================================================
# PART 7: LIVE RETRIEVAL & AUGMENTATION IN CHROMADB
# Explicitly Demonstrating: Ingestion -> Retrieval -> Augmentation
# ==============================================================================
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
# ------------------------------------------------------------------------------
# STAGE 1: INGESTION (Load -> Chunk -> Embed -> Store)
# ------------------------------------------------------------------------------
print("=" * 80)
print("STAGE 1: DATA INGESTION PIPELINE")
print("=" * 80)
# 1A. Load Raw Document
document_text = """
Our refund policy allows customers to return products within 30 days of purchase.
The item must be in its original packaging and unused condition.
Digital downloads are excluded from this policy and are non-refundable after access.
To initiate a return, contact support@example.com with your order number.
Shipping takes 3-5 business days for standard delivery across the country.
Express delivery is available in metro cities and takes 1-2 business days for $15 extra.
Free shipping applies to all orders above $100.
"""
# 1B. Chunking Setup A: Large Naive Chunks (500 chars, 0 overlap)
splitter_bad = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
chunks_bad = splitter_bad.split_text(document_text)
print(f"\n[SETUP A] Generated {len(chunks_bad)} Chunks (chunk_size=500, overlap=0):")
for i, chunk in enumerate(chunks_bad):
    print(f"  Chunk {i} ({len(chunk)} chars): {repr(chunk[:75])}...")
# 1C. Chunking Setup B: Optimal Recursive Chunks (180 chars, 30 overlap)
splitter_good = RecursiveCharacterTextSplitter(chunk_size=180, chunk_overlap=30)
chunks_good = splitter_good.split_text(document_text)
print(f"\n[SETUP B] Generated {len(chunks_good)} Chunks (chunk_size=180, overlap=30):")
for i, chunk in enumerate(chunks_good):
    print(f"  Chunk {i} ({len(chunk)} chars): {repr(chunk[:75])}...")
# 1D. Embed & Store into ChromaDB Collections
model = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.Client()
# Clean up previous collections to allow safe re-running in Jupyter
for name in ["setup_a_large_naive", "setup_b_optimal_recursive"]:
    try:
        client.delete_collection(name)
    except Exception:
        pass
# Index Setup A
col_bad = client.create_collection("setup_a_large_naive")
vecs_bad = model.encode(chunks_bad).tolist()
col_bad.add(
    documents=chunks_bad,
    embeddings=vecs_bad,
    ids=[f"bad_{i}" for i in range(len(chunks_bad))]
)
# Index Setup B
col_good = client.create_collection("setup_b_optimal_recursive")
vecs_good = model.encode(chunks_good).tolist()
col_good.add(
    documents=chunks_good,
    embeddings=vecs_good,
    ids=[f"good_{i}" for i in range(len(chunks_good))]
)
print("\n✅ Ingestion complete: Chunks embedded and indexed into ChromaDB collections!")
# ------------------------------------------------------------------------------
# STAGE 2: RETRIEVAL (Query -> Embed Query -> Nearest Neighbor Search)
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("STAGE 2: RETRIEVAL (Vector Similarity Search)")
print("=" * 80)
user_query = "How do I return a digital download?"
query_vec = model.encode([user_query]).tolist()
# Retrieve Top-1 from Setup A
res_bad = col_bad.query(query_embeddings=query_vec, n_results=1)
retrieved_chunk_a = res_bad['documents'][0][0].strip()
distance_a = res_bad['distances'][0][0]
# Retrieve Top-1 from Setup B
res_good = col_good.query(query_embeddings=query_vec, n_results=1)
retrieved_chunk_b = res_good['documents'][0][0].strip()
distance_b = res_good['distances'][0][0]
print(f"User Query: '{user_query}'")
print(f"\n❌ Setup A Retrieved Distance: {distance_a:.4f} (Weaker match due to noise)")
print(f"Retrieved Text A:\n{retrieved_chunk_a}")
print(f"\n✅ Setup B Retrieved Distance: {distance_b:.4f} (Closer semantic match!)")
print(f"Retrieved Text B:\n{retrieved_chunk_b}")
# ------------------------------------------------------------------------------
# STAGE 3: AUGMENTATION (Injecting Retrieved Context into the LLM Prompt)
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("STAGE 3: AUGMENTATION (Constructing the Final LLM Prompt)")
print("=" * 80)
# We inject the retrieved chunk into the prompt template sent to GPT-4/Claude
augmented_prompt_a = f"""You are a helpful customer support AI. Use ONLY the context below to answer.
Context:
{retrieved_chunk_a}
Question: {user_query}
Answer:"""
augmented_prompt_b = f"""You are a helpful customer support AI. Use ONLY the context below to answer.
Context:
{retrieved_chunk_b}
Question: {user_query}
Answer:"""
print("AUGMENTED PROMPT SENT TO LLM (SETUP B - Optimal):")
print("-" * 60)
print(augmented_prompt_b)
print("-" * 60)
print(f"Total Characters in Setup A Prompt : {len(augmented_prompt_a)} (Bloated)")
print(f"Total Characters in Setup B Prompt : {len(augmented_prompt_b)} (Lean & Accurate)")


STAGE 1: DATA INGESTION PIPELINE

[SETUP A] Generated 2 Chunks (chunk_size=500, overlap=0):
  Chunk 0 (469 chars): 'Our refund policy allows customers to return products within 30 days of pur'...
  Chunk 1 (47 chars): 'Free shipping applies to all orders above $100.'...

[SETUP B] Generated 4 Chunks (chunk_size=180, overlap=30):
  Chunk 0 (146 chars): 'Our refund policy allows customers to return products within 30 days of pur'...
  Chunk 1 (158 chars): 'Digital downloads are excluded from this policy and are non-refundable afte'...
  Chunk 2 (163 chars): 'Shipping takes 3-5 business days for standard delivery across the country.\n'...
  Chunk 3 (47 chars): 'Free shipping applies to all orders above $100.'...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11783.36it/s]



✅ Ingestion complete: Chunks embedded and indexed into ChromaDB collections!

STAGE 2: RETRIEVAL (Vector Similarity Search)
User Query: 'How do I return a digital download?'

❌ Setup A Retrieved Distance: 1.0815 (Weaker match due to noise)
Retrieved Text A:
Our refund policy allows customers to return products within 30 days of purchase.
The item must be in its original packaging and unused condition.
Digital downloads are excluded from this policy and are non-refundable after access.
To initiate a return, contact support@example.com with your order number.
Shipping takes 3-5 business days for standard delivery across the country.
Express delivery is available in metro cities and takes 1-2 business days for $15 extra.

✅ Setup B Retrieved Distance: 0.5795 (Closer semantic match!)
Retrieved Text B:
Digital downloads are excluded from this policy and are non-refundable after access.
To initiate a return, contact support@example.com with your order number.

STAGE 3: AUGMENTATION (Constru